# 🕐 SARIMA Time-Series-Workflow
**Generisches Notebook für saisonale Zeitreihenanalyse**

Dieses Notebook implementiert folgenden Workflow:

1. **Daten laden** – CSV einlesen, Datumsindex erzeugen
2. **Sichtprüfung** – Trend, Saisonalität, wachsende Amplitude?
3. **Log-Transformation** – falls Varianz mit dem Level wächst
4. **Stationarität** – ADF-Test → Differenzierung (d, D)
5. **ACF / PACF** – p, q, P, Q ablesen
6. **Modell schätzen** – SARIMA fitten, AIC vergleichen
7. **Residuen-Diagnose** – Ljung-Box-Test
8. **Prognose** – Forecast + Konfidenzintervall

---

## 0 – Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 100})

---
## 1 – Daten laden

> **Hier den gewünschten Datensatz auswählen.**
> Einfach die Variable `DATENSATZ` auf einen der vier Namen setzen.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  >>> HIER DATENSATZ WÄHLEN <<<                       ║
# ║  Optionen: "AirPassengers", "UKgas",                 ║
# ║            "USAccDeaths", "Temperatur"                ║
# ╚══════════════════════════════════════════════════════╝

DATENSATZ = "zrsyn"  # <-- ändern!

In [ ]:
def lade_datensatz(name: str) -> tuple[pd.Series, int]:
    """Lädt einen der vier Datensätze und gibt (Serie, Saisonperiode) zurück."""

    if name == "AirPassengers":
        df = pd.read_csv("Datasets/AirPassengers.csv")
        idx = pd.to_datetime(df["Month"])
        ts = pd.Series(df["Passengers"].values, index=idx, name="Passengers")
        ts.index.freq = "MS"
        m = 12  # monatlich

    elif name == "UKgas":
        df = pd.read_csv("UKgas.csv")
        # Dezimaljahre → Quartals-Zeitstempel
        idx = pd.PeriodIndex.from_fields(
            year=df["time"].astype(int),
            quarter=((df["time"] % 1) * 4 + 1).astype(int),
            freq="Q",
        ).to_timestamp()
        ts = pd.Series(df["value"].values, index=idx, name="UKgas")
        ts.index.freq = "QS"
        m = 4  # quartalsweise

    elif name == "USAccDeaths":
        df = pd.read_csv("USAccDeaths.csv")
        start_year = int(df["time"].iloc[0])
        idx = pd.date_range(start=f"{start_year}-01", periods=len(df), freq="MS")
        ts = pd.Series(df["value"].values, index=idx, name="USAccDeaths")
        m = 12  # monatlich

    elif name == "Temperatur":
        df = pd.read_csv("syn_temperatur_df.csv")
        idx = pd.to_datetime(df["Timestamp"])
        ts = pd.Series(df["Temperature (°C)"].values, index=idx, name="Temperatur")
        ts.index.freq = "h"
        m = 24  # stündlich → Tages-Saisonalität
    
    elif name == "zrsyn":
        df = pd.read_csv("Datasets/zrsyn.csv")
        idx = pd.to_datetime(df["Period"])
        ts = pd.Series(df["Value"].values, index=idx, name="werte")
        ts.index.freq = "ME"
        m = 12  # stündlich → Tages-Saisonalität

    else:
        raise ValueError(f"Unbekannter Datensatz: {name}")

    print(f"Datensatz: {name}")
    print(f"Zeitraum:  {ts.index[0].date()} bis {ts.index[-1].date()}")
    print(f"Anzahl:    {len(ts)} Beobachtungen")
    print(f"Frequenz:  m = {m}")
    return ts, m

ts_orig, m = lade_datensatz(DATENSATZ)
ts_orig.head()

---
## 2 – Sichtprüfung

Drei Fragen:
- Gibt es einen **Trend** (steigende/fallende Tendenz)?
- Gibt es **Saisonalität** (wiederkehrendes Muster alle m Perioden)?
- **Wächst die Amplitude** mit dem Level? → Log-Transformation nötig

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Rohdaten
axes[0].plot(ts_orig, color="steelblue")
axes[0].set_title("Rohdaten – Zeitreihe")
axes[0].set_ylabel(ts_orig.name)

# Saisonale Zerlegung (additiv als erste Orientierung)
try:
    decomp = seasonal_decompose(ts_orig, model="additive", period=m)
    decomp.seasonal.plot(ax=axes[1], color="coral")
    axes[1].set_title("Saisonale Komponente (additive Zerlegung)")
except Exception as e:
    axes[1].text(0.5, 0.5, f"Zerlegung nicht möglich:\n{e}",
                 transform=axes[1].transAxes, ha="center")

plt.tight_layout()
plt.show()

print("\n→ Frage: Wächst die Schwankungsbreite mit dem Niveau?")
print("  Wenn ja → Log-Transformation in Schritt 2a aktivieren.")

---
## 2a – Log-Transformation (optional)

Wenn die **Amplitude mit dem Level wächst** (z.B. AirPassengers),
stabilisiert `np.log()` die Varianz.

> **Regel:** `LOG_TRANSFORM = True` setzen, wenn die Schwankungen
> am Ende der Reihe deutlich größer sind als am Anfang.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  >>> LOG-TRANSFORMATION EIN/AUS <<<                  ║
# ╚══════════════════════════════════════════════════════╝

LOG_TRANSFORM = False   # <-- auf False setzen, wenn nicht nötig

if LOG_TRANSFORM:
    ts = np.log(ts_orig)
    print("✔ Log-Transformation angewendet.")
else:
    ts = ts_orig.copy()
    print("– Keine Transformation, Originaldaten werden verwendet.")

ts.plot(title="Arbeitszeitreihe (ggf. log-transformiert)", color="steelblue")
plt.ylabel(ts.name)
plt.show()

---
## 3 – ADF-Test (Augmented Dickey-Fuller)

Nullhypothese: Die Zeitreihe hat eine **Einheitswurzel** (= nicht stationär).

- **p < 0.05** → stationär ✓
- **p ≥ 0.05** → nicht stationär → Differenzierung nötig

In [ ]:
def adf_bericht(serie: pd.Series, label: str = "") -> float:
    """Führt den ADF-Test durch und gibt den p-Wert zurück."""
    result = adfuller(serie.dropna(), autolag="AIC")
    p = result[1]
    stat = result[0]
    symbol = "✔ stationär" if p < 0.05 else "✗ NICHT stationär"
    print(f"ADF-Test {label}")
    print(f"  Teststatistik = {stat:.4f}")
    print(f"  p-Wert        = {p:.6f}")
    print(f"  Ergebnis      → {symbol}\n")
    return p

p_original = adf_bericht(ts, "(Arbeitszeitreihe)")

---
## 4 – Differenzierung testen

Wir probieren systematisch:
- **Einfache Differenz** (d=1): entfernt linearen Trend
- **Saisonale Differenz** (D=1): entfernt saisonales Muster
- **Beides** (d=1, D=1): falls nötig

Der ADF-Test bestätigt, welche Kombination Stationarität herstellt.

In [ ]:
# Verschiedene Differenzierungen durchprobieren
varianten = {}

# d=1
ts_d1 = ts.diff().dropna()
varianten["d=1, D=0"] = (ts_d1, 1, 0)

# D=1 (saisonale Differenz)
ts_D1 = ts.diff(m).dropna()
varianten["d=0, D=1"] = (ts_D1, 0, 1)

# d=1 + D=1
ts_d1D1 = ts.diff(m).diff().dropna()
varianten["d=1, D=1"] = (ts_d1D1, 1, 1)

print("="*50)
ergebnisse = {}
for label, (serie, d_val, D_val) in varianten.items():
    p = adf_bericht(serie, f"({label})")
    ergebnisse[label] = (p, d_val, D_val, serie)

# Beste Variante auswählen (stationär + möglichst wenig Differenzierung)
stationaere = {k: v for k, v in ergebnisse.items() if v[0] < 0.05}

if p_original < 0.05:
    d, D = 0, 0
    ts_diff = ts
    print("→ Bereits stationär! d=0, D=0")
elif stationaere:
    # Wähle die mit geringstem d+D
    beste = min(stationaere.items(), key=lambda x: x[1][1] + x[1][2])
    d, D = beste[1][1], beste[1][2]
    ts_diff = beste[1][3]
    print(f"→ Gewählt: {beste[0]} (p = {beste[1][0]:.6f})")
else:
    # Fallback
    d, D = 1, 1
    ts_diff = ts_d1D1
    print("→ Keine Variante klar stationär – verwende d=1, D=1 als Ausgangspunkt.")

print(f"\n★ Differenzierungs-Ordnung:  d = {d},  D = {D}")

In [ ]:
# Differenzierte Reihe plotten
ts_diff.plot(title=f"Differenzierte Reihe (d={d}, D={D})", color="steelblue")
plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.ylabel("Differenz")
plt.show()

---
## 5 – ACF / PACF ablesen

Auf der **differenzierten Reihe** lesen wir die Ordnungen ab:

| Plot | Schaut auf | Bestimmt |
|------|-----------|----------|
| **ACF** | Autokorrelation bei Lag k | **q** (MA) und **Q** (saisonales MA) |
| **PACF** | Partielle Autokorrelation | **p** (AR) und **P** (saisonales AR) |

**Faustregeln:**
- Signifikanter Spike bei Lag 1–3 im PACF → p = Anzahl Spikes
- Signifikanter Spike bei Lag 1–3 im ACF → q = Anzahl Spikes
- Spike bei Lag **m** (z.B. 12) → P=1 oder Q=1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

lags = min(4 * m, len(ts_diff) // 2 - 1)  # sinnvolle Lag-Anzahl

plot_acf(ts_diff, lags=lags, ax=axes[0], title="ACF")
plot_pacf(ts_diff, lags=lags, ax=axes[1], title="PACF", method="ywm")

# Saisonale Lags markieren
for ax in axes:
    for k in range(m, lags + 1, m):
        ax.axvline(k, color="coral", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.show()

print(f"Rote Linien = saisonale Lags (alle {m} Perioden)")
print("\n→ Spikes außerhalb des Konfidenz-Bandes ablesen,")
print("  dann p, q, P, Q in der nächsten Zelle eintragen.")

---
## 6 – Modell schätzen

Basierend auf ACF/PACF die Ordnungen eintragen.
Mehrere Kandidaten werden verglichen – das Modell mit dem **niedrigsten AIC** gewinnt.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  >>> KANDIDATEN-MODELLE DEFINIEREN <<<               ║
# ║  Format: (p, d, q) x (P, D, Q, m)                   ║
# ╚══════════════════════════════════════════════════════╝

kandidaten = [
    # (p, d, q, P, D, Q)  – m wird automatisch gesetzt
    (1, d, 1, 1, D, 1),
    (0, d, 1, 0, D, 1),
    (1, d, 0, 1, D, 0),
    (2, d, 1, 1, D, 1),
    (1, d, 1, 0, D, 1),
    (1, d, 1, 1, D, 0),
]

print(f"Saisonperiode m = {m}")
print(f"Differenzierung: d = {d}, D = {D}")
print("-" * 60)

ergebnis_modelle = []
for (p, d_, q, P, D_, Q) in kandidaten:
    order = (p, d_, q)
    seasonal = (P, D_, Q, m)
    label = f"SARIMA{order}x{seasonal}"
    try:
        modell = SARIMAX(ts, order=order, seasonal_order=seasonal,
                         enforce_stationarity=False,
                         enforce_invertibility=False)
        fit = modell.fit(disp=False, maxiter=200)
        ergebnis_modelle.append((label, fit, fit.aic, fit.bic))
        print(f"  {label:<42s}  AIC={fit.aic:>10.2f}  BIC={fit.bic:>10.2f}")
    except Exception as e:
        print(f"  {label:<42s}  ✗ Fehler: {e}")

# Bestes Modell nach AIC
ergebnis_modelle.sort(key=lambda x: x[2])
bestes_label, bestes_modell, bester_aic, bester_bic = ergebnis_modelle[0]
print(f"\n★ Bestes Modell: {bestes_label}  (AIC = {bester_aic:.2f})")

---
## 7 – Residuen-Diagnose

Ein gutes Modell hat Residuen, die **weißem Rauschen** ähneln:
- Keine signifikanten Autokorrelationen (Ljung-Box p > 0.05)
- Annähernd normalverteilt
- Kein erkennbares Muster

In [ ]:
fig = bestes_modell.plot_diagnostics(figsize=(13, 8))
plt.suptitle(f"Residuen-Diagnose: {bestes_label}", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box-Test auf die Residuen
lb = acorr_ljungbox(bestes_modell.resid.dropna(), lags=[m, 2*m], return_df=True)
print("Ljung-Box-Test (H₀: keine Autokorrelation in Residuen):")
print(lb.to_string())
print()

alle_ok = (lb["lb_pvalue"] > 0.05).all()
if alle_ok:
    print("✔ Residuen zeigen KEINE signifikante Autokorrelation → Modell OK!")
else:
    print("✗ Signifikante Autokorrelation gefunden!")
    print("  → Evtl. p, q, P, Q anpassen und Schritt 5–7 wiederholen.")

---
## 8 – Prognose

Forecast erzeugen + ggf. Rücktransformation (exp) falls Log angewendet wurde.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  >>> PROGNOSEHORIZONT EINSTELLEN <<<                 ║
# ╚══════════════════════════════════════════════════════╝

HORIZONT = 2 * m   # z.B. 2 volle Saison-Zyklen voraus

forecast = bestes_modell.get_forecast(steps=HORIZONT)
pred_mean = forecast.predicted_mean
pred_ci = forecast.conf_int(alpha=0.05)

# Rücktransformation, falls Log
if LOG_TRANSFORM:
    pred_mean = np.exp(pred_mean)
    pred_ci = np.exp(pred_ci)
    plot_ts = ts_orig  # Original zum Plotten
else:
    plot_ts = ts_orig

# Plot
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(plot_ts, label="Beobachtet", color="steelblue")
ax.plot(pred_mean.index, pred_mean, label="Prognose", color="coral", linewidth=2)
ax.fill_between(
    pred_ci.index,
    pred_ci.iloc[:, 0],
    pred_ci.iloc[:, 1],
    color="coral", alpha=0.2, label="95%-Konfidenzintervall",
)
ax.set_title(f"Prognose mit {bestes_label}", fontsize=14)
ax.set_ylabel(ts_orig.name)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nPrognose für {HORIZONT} Perioden ab {pred_mean.index[0].date()}:")
print(pred_mean.round(2).to_string())

---
## Zusammenfassung

| Schritt | Ergebnis |
|---------|----------|
| Datensatz | → `DATENSATZ` Variable oben |
| Log-Transformation | → `LOG_TRANSFORM` |
| Differenzierung | → `d`, `D` automatisch bestimmt |
| Modell | → bestes AIC aus Kandidatenliste |
| Residuen | → Ljung-Box-Test |
| Prognose | → `HORIZONT` Perioden |

**Zum Ausprobieren:** Einfach `DATENSATZ` oben ändern und alles neu ausführen!